# Phase 04 Colab Notebook: Vision Backbone Feature Extraction & OOD Statistics

This notebook executes Phase 04 feature extraction for **crop-prognosis** in Google Colab.
It loads the frozen DINOv2 vision backbone (`facebook/dinov2-small`), processes all static in-the-wild images and ETH Zurich wheat leaf time-series photos, extracts CLS tokens, mean patch tokens, and patch token grids (`32x32`), computes shrinkage-regularized covariance OOD statistics, and caches feature embeddings as sharded `.safetensors` files on Google Drive.

In [ ]:
# Step 1: Install Dependencies & Check Hardware Acceleration
!pip install -q safetensors transformers torch pandas pydantic scikit-learn pillow matplotlib

import torch
print('PyTorch Version:', torch.__version__)
print('CUDA Available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU Device Name:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU mode (Stub model will be used for testing if no GPU).')

In [ ]:
# Step 2: Configure Workspace & Python Import Paths
import os
import sys
from pathlib import Path

if os.path.exists('/content/drive'):
    drive_root = '/content/drive/MyDrive/crop_prognosis_data'
    os.makedirs(drive_root, exist_ok=True)
    os.environ['DATA_ROOT'] = drive_root
else:
    os.environ['DATA_ROOT'] = '/content/data'
print('DATA_ROOT set to:', os.environ['DATA_ROOT'])

candidates = [os.getcwd(), '/content/PlantDiseaseSeverity', '/content/crop-prognosis']
found_repo = False
for path in candidates:
    if os.path.exists(os.path.join(path, '.git')) and os.path.exists(os.path.join(path, 'ml')):
        os.chdir(path)
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"Using project repository root at: {path}")
        found_repo = True
        break

if not found_repo:
    print("Standalone Colab mode: Writing updated 'ml' package files to /content/ml...")
    os.makedirs('/content/ml/utils', exist_ok=True)
    os.makedirs('/content/ml/data', exist_ok=True)
    os.makedirs('/content/ml/features', exist_ok=True)
    os.makedirs('/content/ml/configs', exist_ok=True)
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

    Path('/content/ml/__init__.py').write_text('')
    Path('/content/ml/utils/__init__.py').write_text('')
    Path('/content/ml/data/__init__.py').write_text('')
    Path('/content/ml/features/__init__.py').write_text('')

    Path('/content/ml/utils/paths.py').write_text('"""Centralized project path management.\n\nResolves paths relative to the repository root locally or using the environment variable DATA_ROOT\nfor seamless execution in Google Colab with Google Drive mounts.\n"""\n\nimport os\nfrom pathlib import Path\nfrom typing import NamedTuple\n\nclass ProjectPaths(NamedTuple):\n    project_root: Path\n    data_root: Path\n    raw_data: Path\n    interim_data: Path\n    processed_data: Path\n    models_dir: Path\n    reports_dir: Path\n    figures_dir: Path\n    tables_dir: Path\n    configs_dir: Path\n    docs_dir: Path\n\ndef get_paths() -> ProjectPaths:\n    """Resolve and return all project directory paths.\n    \n    Reads DATA_ROOT from environment if present (e.g., in Google Colab),\n    otherwise defaults to data/ inside the repository root.\n    """\n    # ml/utils/paths.py is 2 levels below repository root\n    project_root = Path(__file__).resolve().parent.parent.parent\n    \n    env_data_root = os.getenv("DATA_ROOT")\n    if env_data_root:\n        data_root = Path(env_data_root).resolve()\n    else:\n        data_root = (project_root / "data").resolve()\n        \n    raw_data = data_root / "raw"\n    interim_data = data_root / "interim"\n    processed_data = data_root / "processed"\n    models_dir = project_root / "models"\n    reports_dir = project_root / "reports"\n    figures_dir = reports_dir / "figures"\n    tables_dir = reports_dir / "tables"\n    configs_dir = project_root / "ml" / "configs"\n    docs_dir = project_root / "docs"\n    \n    # Ensure directories exist\n    for path in [data_root, raw_data, interim_data, processed_data, models_dir, reports_dir, figures_dir, tables_dir]:\n        path.mkdir(parents=True, exist_ok=True)\n        \n    return ProjectPaths(\n        project_root=project_root,\n        data_root=data_root,\n        raw_data=raw_data,\n        interim_data=interim_data,\n        processed_data=processed_data,\n        models_dir=models_dir,\n        reports_dir=reports_dir,\n        figures_dir=figures_dir,\n        tables_dir=tables_dir,\n        configs_dir=configs_dir,\n        docs_dir=docs_dir,\n    )\n')
    Path('/content/ml/utils/logging_utils.py').write_text('"""Logging configuration helper for standard structured logging across crop-prognosis."""\n\nimport logging\nimport sys\nfrom typing import Optional\n\ndef get_logger(name: str, level: int = logging.INFO, log_file: Optional[str] = None) -> logging.Logger:\n    """Create or retrieve a configured Python standard logger.\n    \n    Args:\n        name: Name of the logger (typically __name__).\n        level: Logging level (e.g. logging.INFO, logging.DEBUG).\n        log_file: Optional file path to store log output alongside stderr.\n        \n    Returns:\n        Configured logging.Logger object.\n    """\n    logger = logging.getLogger(name)\n    logger.setLevel(level)\n    \n    # Avoid duplicate handlers if logger is already configured\n    if logger.handlers:\n        return logger\n        \n    formatter = logging.Formatter(\n        fmt="[%(asctime)s] [%(levelname)s] [%(name)s]: %(message)s",\n        datefmt="%Y-%m-%d %H:%M:%S"\n    )\n    \n    # Console handler\n    console_handler = logging.StreamHandler(sys.stderr)\n    console_handler.setFormatter(formatter)\n    console_handler.setLevel(level)\n    logger.addHandler(console_handler)\n    \n    # Optional file handler\n    if log_file:\n        file_handler = logging.FileHandler(log_file, encoding="utf-8")\n        file_handler.setFormatter(formatter)\n        file_handler.setLevel(level)\n        logger.addHandler(file_handler)\n        \n    return logger\n')
    Path('/content/ml/utils/io.py').write_text('"""Safe file input/output utilities enforcing secure serialization.\n\nSupports JSON, YAML, and Parquet. Explicitly blocks and refuses any pickle operations\nto protect against unsafe deserialization vulnerabilities.\n"""\n\nimport json\nfrom pathlib import Path\nfrom typing import Any, Union\nimport yaml\nimport pandas as pd\n\nclass PickleRefusedError(ValueError):\n    """Exception raised when an unsafe pickle operation is attempted."""\n    pass\n\ndef _check_not_pickle(filepath: Union[str, Path]) -> None:\n    """Refuse files with pickle extensions."""\n    path = Path(filepath)\n    ext = path.suffix.lower()\n    if ext in [".pkl", ".pickle", ".pck", ".pd"]:\n        raise PickleRefusedError(\n            f"Security policy error: Refusing to interact with pickle file \'{path}\'. "\n            "Use JSON, YAML, Safetensors, or Parquet for model storage and data persistence."\n        )\n\ndef read_json(filepath: Union[str, Path]) -> Any:\n    """Safely read and parse a JSON file."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    with path.open("r", encoding="utf-8") as f:\n        return json.load(f)\n\ndef write_json(data: Any, filepath: Union[str, Path], indent: int = 2) -> None:\n    """Safely write data to a JSON file."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as f:\n        json.dump(data, f, indent=indent, ensure_ascii=False)\n\ndef read_yaml(filepath: Union[str, Path]) -> dict[str, Any]:\n    """Safely read and parse a YAML file using yaml.safe_load."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    with path.open("r", encoding="utf-8") as f:\n        return yaml.safe_load(f) or {}\n\ndef write_yaml(data: dict[str, Any], filepath: Union[str, Path]) -> None:\n    """Safely write data to a YAML file using yaml.safe_dump."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    with path.open("w", encoding="utf-8") as f:\n        yaml.safe_dump(data, f, sort_keys=False)\n\ndef read_parquet(filepath: Union[str, Path]) -> pd.DataFrame:\n    """Safely read a Parquet file into a pandas DataFrame."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    return pd.read_parquet(path)\n\ndef write_parquet(df: pd.DataFrame, filepath: Union[str, Path]) -> None:\n    """Safely write a pandas DataFrame to a Parquet file."""\n    _check_not_pickle(filepath)\n    path = Path(filepath)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    df.to_parquet(path, index=False)\n\ndef load_pickle(*args: Any, **kwargs: Any) -> None:\n    """Explicitly blocked function raising PickleRefusedError."""\n    raise PickleRefusedError(\n        "Security policy error: Pickle loading is strictly forbidden in crop-prognosis. "\n        "Use read_json, read_yaml, read_parquet, or safetensors."\n    )\n\ndef dump_pickle(*args: Any, **kwargs: Any) -> None:\n    """Explicitly blocked function raising PickleRefusedError."""\n    raise PickleRefusedError(\n        "Security policy error: Pickle saving is strictly forbidden in crop-prognosis. "\n        "Use write_json, write_yaml, write_parquet, or safetensors."\n    )\n')
    Path('/content/ml/data/provenance.py').write_text('"""Data Provenance and Perceptual Image Deduplication Tool.\n\nCalculates perceptual hashes (dHash/pHash) for image collections, clusters near-duplicate\nimages across static datasets, and exports a standardized static manifest to\ndata/processed/static_manifest.parquet.\n"""\n\nimport sys\nfrom pathlib import Path\nfrom typing import Optional, Union\nfrom PIL import Image\nimport numpy as np\nimport pandas as pd\nfrom pydantic import BaseModel, Field\n\n# Ensure project root is in sys.path for direct script execution\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import write_parquet\n\nlogger = get_logger(__name__)\n\nclass ManifestRecord(BaseModel):\n    """Schema definition for data/processed/static_manifest.parquet records."""\n    image_id: str = Field(..., description="Unique image identifier.")\n    source: str = Field(..., description="Source dataset name.")\n    path: str = Field(..., description="Relative filepath.")\n    licence: str = Field(..., description="Captured dataset license.")\n    width: int = Field(..., description="Image width in pixels.")\n    height: int = Field(..., description="Image height in pixels.")\n    hash: str = Field(..., description="Hexadecimal perceptual dHash string.")\n    cluster_id: int = Field(..., description="Near-duplicate cluster group ID.")\n\ndef compute_dhash(image: Union[Image.Image, str, Path], hash_size: int = 8) -> str:\n    """Compute difference hash (dHash) for a PIL image or image file path.\n    \n    Args:\n        image: Input PIL Image instance or path to an image file.\n        hash_size: Grid resolution (default 8 yields 64-bit hash).\n        \n    Returns:\n        64-character binary/hex string representation of the hash.\n    """\n    if isinstance(image, (str, Path)):\n        image = Image.open(image)\n    resized = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)\n    pixels = np.array(resized)\n    \n    # Calculate difference between adjacent horizontal pixels\n    diff = pixels[:, 1:] > pixels[:, :-1]\n    \n    # Convert boolean array to hex string\n    binary_str = "".join(["1" if b else "0" for b in diff.flatten()])\n    hex_str = f"{int(binary_str, 2):016x}"\n    return hex_str\n\ndef hamming_distance(hex_hash1: str, hex_hash2: str) -> int:\n    """Compute Hamming distance (number of differing bits) between two hex hashes."""\n    b1 = bin(int(hex_hash1, 16))[2:].zfill(64)\n    b2 = bin(int(hex_hash2, 16))[2:].zfill(64)\n    return sum(c1 != c2 for c1, c2 in zip(b1, b2))\n\ndef cluster_near_duplicates(hashes: list[str], max_distance: int = 5) -> list[int]:\n    """Group image indices into clusters based on Hamming distance threshold."""\n    n = len(hashes)\n    cluster_ids = [-1] * n\n    current_cluster = 0\n    \n    for i in range(n):\n        if cluster_ids[i] != -1:\n            continue\n        cluster_ids[i] = current_cluster\n        for j in range(i + 1, n):\n            if cluster_ids[j] == -1 and hamming_distance(hashes[i], hashes[j]) <= max_distance:\n                cluster_ids[j] = current_cluster\n        current_cluster += 1\n        \n    return cluster_ids\n\ndef build_static_manifest(data_root: Optional[Path] = None, max_distance: int = 5) -> pd.DataFrame:\n    """Build static manifest Parquet file indexing all available dataset images."""\n    if data_root is None:\n        paths = get_paths()\n        data_root = paths.data_root\n        \n    records = []\n    logger.info(f"Scanning images in {data_root / \'raw\'} for provenance manifest...")\n    \n    # Image extensions\n    image_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff"}\n    image_files = [p for p in (data_root / "raw").rglob("*") if p.suffix.lower() in image_exts]\n    \n    for idx, file_path in enumerate(image_files):\n        try:\n            with Image.open(file_path) as img:\n                w, h = img.size\n                h_val = compute_dhash(img)\n                \n            source_name = file_path.relative_to(data_root / "raw").parts[0]\n            records.append({\n                "image_id": f"{source_name}_{idx:06d}",\n                "source": source_name,\n                "path": str(file_path.relative_to(data_root)),\n                "licence": "CC BY 4.0" if "eth" in source_name else "Unknown",\n                "width": w,\n                "height": h,\n                "hash": h_val,\n            })\n        except Exception as e:\n            logger.warning(f"Failed to process image {file_path}: {e}")\n\n    if not records:\n        logger.warning("No image files found. Returning empty manifest DataFrame.")\n        df = pd.DataFrame(columns=["image_id", "source", "path", "licence", "width", "height", "hash", "cluster_id"])\n    else:\n        df = pd.DataFrame(records)\n        cluster_ids = cluster_near_duplicates(df["hash"].tolist(), max_distance=max_distance)\n        df["cluster_id"] = cluster_ids\n        \n    # Save output to static_manifest.parquet\n    output_path = paths.project_root / "data" / "processed" / "static_manifest.parquet"\n    write_parquet(df, output_path)\n    logger.info(f"Exported manifest with {len(df)} records to {output_path}")\n    return df\n\ndef main() -> int:\n    build_static_manifest()\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/data/build_static_manifest.py').write_text('"""Unified Static In-The-Wild Dataset Manifest Builder.\n\nScans in-the-wild disease datasets (PlantSeg, Coffee Rust, LeafNet75, PlantVillage),\nstandardizes lesion masks to binary PNG (0/1), verifies mask integrity, calculates\nperceptual hashes (dhash) and near-duplicate cluster IDs, and exports data/processed/static/manifest.parquet.\n"""\n\nimport sys\nimport json\nimport argparse\nfrom pathlib import Path\nfrom typing import Any, Optional, Tuple\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import write_parquet\nfrom ml.data.provenance import compute_dhash, cluster_near_duplicates\n\nlogger = get_logger(__name__)\n\ndef validate_mask_integrity(image_path: Path, mask_path: Path) -> Tuple[bool, str]:\n    """Verify single-channel mask dimension match and binary 0/1 pixel values."""\n    if not image_path.exists():\n        return False, "Image file missing"\n    if not mask_path.exists():\n        return False, "Mask file missing"\n    try:\n        img = Image.open(image_path)\n        mask = Image.open(mask_path)\n        img_w, img_h = img.size\n        mask_w, mask_h = mask.size\n\n        if (img_w, img_h) != (mask_w, mask_h):\n            return False, f"Dimension mismatch: image ({img_w}x{img_h}) vs mask ({mask_w}x{mask_h})"\n\n        mask_arr = np.array(mask)\n        unique_vals = np.unique(mask_arr)\n        if not set(unique_vals).issubset({0, 1}):\n            return False, f"Non-binary mask values found: {unique_vals}"\n\n        return True, "Valid"\n    except Exception as e:\n        return False, f"Error validating mask: {str(e)}"\n\ndef generate_synthetic_static_dataset(raw_dir: Path) -> None:\n    """Generate lightweight synthetic static datasets when raw files are missing."""\n    logger.info(f"Generating synthetic static datasets at {raw_dir}...")\n\n    # 1. Synthetic PlantSeg\n    plantseg_dir = raw_dir / "plantseg" / "images"\n    plantseg_dir.mkdir(parents=True, exist_ok=True)\n    for i in range(1, 11):\n        img = Image.fromarray(np.random.randint(50, 200, (120, 160, 3), dtype=np.uint8))\n        img.save(plantseg_dir / f"plantseg_{i:03d}.jpg")\n        mask = np.zeros((120, 160), dtype=np.uint8)\n        mask[30:70, 40:100] = 1 # Lesion\n        Image.fromarray(mask).save(plantseg_dir / f"plantseg_{i:03d}_mask.png")\n\n    # 2. Synthetic Coffee Rust\n    coffee_img_dir = raw_dir / "coffee_rust" / "02_binary_images" / "base_image"\n    coffee_mask_dir = raw_dir / "coffee_rust" / "02_binary_images" / "golden_standard"\n    coffee_eval_dir = coffee_mask_dir / "100_eval_set"\n    coffee_img_dir.mkdir(parents=True, exist_ok=True)\n    coffee_mask_dir.mkdir(parents=True, exist_ok=True)\n    coffee_eval_dir.mkdir(parents=True, exist_ok=True)\n\n    for i in range(1, 15):\n        img_name = f"coffee_{i:03d}.jpg"\n        img = Image.fromarray(np.random.randint(40, 220, (100, 100, 3), dtype=np.uint8))\n        img.save(coffee_img_dir / img_name)\n\n        mask = np.zeros((100, 100), dtype=np.uint8)\n        mask[20:50, 20:60] = 1\n        if i <= 10:\n            Image.fromarray(mask).save(coffee_mask_dir / f"coffee_{i:03d}_mask.png")\n        else:\n            Image.fromarray(mask).save(coffee_eval_dir / f"coffee_{i:03d}_eval_mask.png")\n\n    logger.info("Synthetic static datasets generated.")\n\ndef process_static_manifest(raw_dir: Optional[Path] = None, processed_dir: Optional[Path] = None) -> pd.DataFrame:\n    """Scan static datasets and assemble unified manifest.parquet."""\n    paths = get_paths()\n    if raw_dir is None:\n        raw_dir = paths.raw_data\n    if processed_dir is None:\n        processed_dir = paths.project_root / "data" / "processed" / "static"\n\n    processed_dir.mkdir(parents=True, exist_ok=True)\n    lesion_mask_out_dir = processed_dir / "lesion_masks"\n    images_out_dir = processed_dir / "images"\n    lesion_mask_out_dir.mkdir(parents=True, exist_ok=True)\n    images_out_dir.mkdir(parents=True, exist_ok=True)\n\n    # Generate synthetic fallback data if missing\n    if not (raw_dir / "plantseg").exists() and not (raw_dir / "coffee_rust").exists():\n        generate_synthetic_static_dataset(raw_dir)\n\n    records = []\n\n    # 1. Process PlantSeg\n    plantseg_raw = raw_dir / "plantseg"\n    for img_path in plantseg_raw.rglob("*.jpg"):\n        stem = img_path.stem\n        if stem.endswith("_mask"):\n            continue\n        mask_path = img_path.parent / f"{stem}_mask.png"\n        if not mask_path.exists():\n            continue\n\n        # Save standard image & mask\n        std_img_path = images_out_dir / f"plantseg_{stem}.jpg"\n        std_mask_path = lesion_mask_out_dir / f"plantseg_{stem}_lesion.png"\n\n        Image.open(img_path).convert("RGB").save(std_img_path)\n        m_arr = np.array(Image.open(mask_path))\n        m_binary = (m_arr > 0).astype(np.uint8)\n        Image.fromarray(m_binary).save(std_mask_path)\n\n        valid, msg = validate_mask_integrity(std_img_path, std_mask_path)\n        dhash = compute_dhash(std_img_path)\n\n        img_obj = Image.open(std_img_path)\n        records.append({\n            "image_id": f"plantseg_{stem}",\n            "source": "plantseg",\n            "rel_image_path": str(std_img_path.relative_to(paths.project_root)),\n            "rel_lesion_mask_path": str(std_mask_path.relative_to(paths.project_root)),\n            "rel_leaf_mask_path": None,\n            "leaf_mask_origin": "none",\n            "crop": "General Plant",\n            "disease": "Various",\n            "licence": "Custom / Zenodo",\n            "width": img_obj.width,\n            "height": img_obj.height,\n            "dhash": dhash,\n            "is_quality_valid": valid,\n            "validation_msg": msg\n        })\n\n    # 2. Process Coffee Rust\n    coffee_raw = raw_dir / "coffee_rust"\n    coffee_imgs = list(coffee_raw.rglob("*.jpg")) + list(coffee_raw.rglob("*.png"))\n    for img_path in coffee_imgs:\n        stem = img_path.stem\n        if "mask" in stem or "eval" in stem or "golden_standard" in str(img_path):\n            continue\n\n        # Look for corresponding mask\n        mask_path = None\n        is_eval = False\n        for m_candidate in coffee_raw.rglob(f"*{stem}*mask.png"):\n            mask_path = m_candidate\n            if "100_eval_set" in str(m_candidate):\n                is_eval = True\n            break\n\n        if mask_path is None or not mask_path.exists():\n            continue\n\n        std_img_path = images_out_dir / f"coffee_{stem}.jpg"\n        std_mask_path = lesion_mask_out_dir / f"coffee_{stem}_lesion.png"\n\n        Image.open(img_path).convert("RGB").save(std_img_path)\n        m_arr = np.array(Image.open(mask_path))\n        m_binary = (m_arr > 0).astype(np.uint8)\n        Image.fromarray(m_binary).save(std_mask_path)\n\n        valid, msg = validate_mask_integrity(std_img_path, std_mask_path)\n        dhash = compute_dhash(std_img_path)\n        img_obj = Image.open(std_img_path)\n\n        records.append({\n            "image_id": f"coffee_{stem}",\n            "source": "coffee_rust",\n            "rel_image_path": str(std_img_path.relative_to(paths.project_root)),\n            "rel_lesion_mask_path": str(std_mask_path.relative_to(paths.project_root)),\n            "rel_leaf_mask_path": None,\n            "leaf_mask_origin": "none",\n            "crop": "Coffee",\n            "disease": "Coffee Leaf Rust",\n            "licence": "CC-BY-4.0",\n            "width": img_obj.width,\n            "height": img_obj.height,\n            "dhash": dhash,\n            "is_quality_valid": valid,\n            "validation_msg": msg,\n            "is_external_eval": is_eval\n        })\n\n    manifest_df = pd.DataFrame(records)\n\n    # Compute near-duplicate cluster IDs\n    if not manifest_df.empty and "dhash" in manifest_df.columns:\n        dhashes = manifest_df["dhash"].tolist()\n        clusters = cluster_near_duplicates(dhashes, max_distance=10)\n        manifest_df["cluster_id"] = clusters\n    else:\n        manifest_df["cluster_id"] = 0\n\n    out_manifest_path = processed_dir / "manifest.parquet"\n    write_parquet(manifest_df, out_manifest_path)\n    logger.info(f"Built unified static manifest with {len(manifest_df)} records at {out_manifest_path}")\n\n    return manifest_df\n\ndef main() -> int:\n    process_static_manifest()\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/data/eth_wheat.py').write_text('"""ETH Zurich Wheat Dataset Parser and Feature Processor.\n\nParses experimental design metadata, hourly weather logs, and symptom segmentation masks.\nOutputs standardized Parquet tables: images_long.parquet, sequences.parquet,\nweather_hourly.parquet, and weather_daily.parquet.\n"""\n\nimport sys\nimport re\nimport argparse\nfrom pathlib import Path\nfrom typing import Any, Optional, Tuple\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import write_parquet\n\nlogger = get_logger(__name__)\n\ndef parse_image_id(filename_stem: str) -> dict[str, Any]:\n    """Parse image ID string into metadata attributes.\n    \n    Expected format: YYYYMMDD_HHMMSS_<plot_id>_<leaf_no>\n    Example: 20230515_103000_ESWW001_L1\n    """\n    pattern = r"^(\\d{8}_\\d{6})_([A-Z0-9]+)_(L\\d+)$"\n    match = re.match(pattern, filename_stem)\n    if match:\n        ts_str, plot_id, leaf_no = match.groups()\n        timestamp = pd.to_datetime(ts_str, format="%Y%m%d_%H%M%S")\n        sample_id = f"{plot_id}_{leaf_no}"\n        return {\n            "image_id": filename_stem,\n            "timestamp": timestamp,\n            "plot_id": plot_id,\n            "leaf_no": leaf_no,\n            "sample_id": sample_id\n        }\n    else:\n        # Generic fallback parsing\n        parts = filename_stem.split("_")\n        sample_id = f"{parts[2] if len(parts)>2 else \'P0\'}_{parts[3] if len(parts)>3 else \'L1\'}"\n        return {\n            "image_id": filename_stem,\n            "timestamp": pd.Timestamp.now(),\n            "plot_id": parts[2] if len(parts)>2 else "P0",\n            "leaf_no": parts[3] if len(parts)>3 else "L1",\n            "sample_id": sample_id\n        }\n\ndef compute_mask_metrics(mask_array: np.ndarray) -> dict[str, Any]:\n    """Compute per-image mask pixel counts, disease ratios, and lesion stats.\n    \n    Mask Pixel Encoding:\n      0: Background\n      1: Healthy Green\n      2: Chlorosis\n      3: Necrosis / Septoria\n      4: Rust Pustules\n      5: Ink Markers\n    """\n    unique_vals, counts = np.unique(mask_array, return_counts=True)\n    val_map = dict(zip(unique_vals, counts))\n\n    healthy_cnt = val_map.get(1, 0)\n    chlorosis_cnt = val_map.get(2, 0)\n    necrosis_cnt = val_map.get(3, 0)\n    pustule_cnt = val_map.get(4, 0)\n\n    diseased_pixels = necrosis_cnt + pustule_cnt\n    leaf_pixels = healthy_cnt + chlorosis_cnt + diseased_pixels\n\n    # Compute lesion connected components (classes 3 and 4)\n    diseased_mask = (mask_array == 3) | (mask_array == 4)\n    n_lesions, mean_lesion_area = _count_connected_lesions(diseased_mask)\n\n    return {\n        "diseased_pixels": int(diseased_pixels),\n        "leaf_pixels_frame": int(leaf_pixels),\n        "necrosis_pixels": int(necrosis_cnt),\n        "pustule_pixels": int(pustule_cnt),\n        "n_lesions": int(n_lesions),\n        "mean_lesion_area": float(mean_lesion_area)\n    }\n\ndef _count_connected_lesions(binary_mask: np.ndarray) -> Tuple[int, float]:\n    """Count connected components and mean component size in a binary mask."""\n    if not np.any(binary_mask):\n        return 0, 0.0\n    try:\n        from scipy.ndimage import label\n        labeled, num_features = label(binary_mask)\n        if num_features == 0:\n            return 0, 0.0\n        sizes = [np.sum(labeled == i) for i in range(1, num_features + 1)]\n        return num_features, float(np.mean(sizes))\n    except Exception:\n        # Simple fallback without scipy\n        total = np.sum(binary_mask)\n        return (1 if total > 0 else 0), float(total)\n\ndef generate_synthetic_eth_dataset(raw_dir: Path) -> None:\n    """Generate a lightweight synthetic ETH dataset structure for testing when real images are missing."""\n    logger.info(f"Generating synthetic ETH dataset tree at {raw_dir}...")\n    meta_dir = raw_dir / "meta"\n    ts_dir = raw_dir / "processed" / "ts"\n    meta_dir.mkdir(parents=True, exist_ok=True)\n    ts_dir.mkdir(parents=True, exist_ok=True)\n\n    # 1. Experimental Design CSV\n    exp_design = []\n    cultivars = ["Arina", "CH_Claro", "Titan", "Runal", "Forel"]\n    treatments = ["FI2", "0FI1", "F0I", "0F0I"]\n    for i in range(1, 11):\n        plot_id = f"ESWW{i:03d}"\n        exp_design.append({\n            "plot_id": plot_id,\n            "cultivar": cultivars[(i - 1) % len(cultivars)],\n            "treatment": treatments[(i - 1) % len(treatments)],\n            "batch": 1 if i <= 5 else 2,\n            "year": 2023,\n            "heading_date": "2023-05-15"\n        })\n    pd.DataFrame(exp_design).to_csv(meta_dir / "experimental_design.csv", index=False)\n\n    # 2. Weather Hourly CSV\n    dates = pd.date_range("2023-05-01", "2023-06-30", freq="h")\n    np.random.seed(42)\n    weather_data = []\n    for d in dates:\n        weather_data.append({\n            "timestamp": d.strftime("%Y-%m-%d %H:%M:%S"),\n            "temperature": 15.0 + 5.0 * np.sin(d.hour / 24.0 * 2 * np.pi) + np.random.normal(0, 0.5),\n            "relative_humidity": 75.0 + 15.0 * np.cos(d.hour / 24.0 * 2 * np.pi) + np.random.normal(0, 1.0)\n        })\n    pd.DataFrame(weather_data).to_csv(meta_dir / "weather_station_hourly.csv", index=False)\n\n    # 3. Create 5 sample sequence folders with synthetic mask images\n    for i in range(1, 6):\n        plot_id = f"ESWW{i:03d}"\n        sample_id = f"{plot_id}_L1"\n        sample_dir = ts_dir / sample_id\n        sample_dir.mkdir(parents=True, exist_ok=True)\n\n        start_time = pd.Timestamp("2023-05-10 10:00:00")\n        for step in range(5):\n            ts = start_time + pd.Timedelta(days=step * 2)\n            ts_str = ts.strftime("%Y%m%d_%H%M%S")\n            img_id = f"{ts_str}_{plot_id}_L1"\n\n            # Create synthetic mask (100x100 pixels)\n            mask = np.ones((100, 100), dtype=np.uint8) # 1: Healthy\n            # Add increasing lesion area (3: Necrosis)\n            lesion_size = 5 + step * 8\n            mask[10:10+lesion_size, 10:10+lesion_size] = 3\n            # Add ink markers (5)\n            mask[0:2, 0:2] = 5\n\n            Image.fromarray(mask).save(sample_dir / f"{img_id}_mask.png")\n            # Create synthetic rgb leaf image\n            rgb = np.stack([mask * 30, mask * 40 + 50, mask * 20], axis=-1).astype(np.uint8)\n            Image.fromarray(rgb).save(sample_dir / f"{img_id}.jpg")\n\n    logger.info("Synthetic ETH dataset generation complete.")\n\ndef process_eth_dataset(raw_dir: Optional[Path] = None, processed_dir: Optional[Path] = None) -> dict[str, Any]:\n    """Parse ETH dataset files and export processed Parquet tables."""\n    paths = get_paths()\n    if raw_dir is None:\n        raw_dir = paths.raw_data / "eth_wheat"\n    if processed_dir is None:\n        processed_dir = paths.project_root / "data" / "processed" / "eth"\n\n    processed_dir.mkdir(parents=True, exist_ok=True)\n\n    meta_dir = raw_dir / "meta"\n    ts_dir = raw_dir / "processed" / "ts"\n\n    # If raw data does not exist, generate synthetic test data\n    if not meta_dir.exists() or not (meta_dir / "experimental_design.csv").exists():\n        logger.warning("ETH metadata CSV files not found. Creating synthetic ETH tree...")\n        generate_synthetic_eth_dataset(raw_dir)\n\n    # 1. Read Experimental Design\n    exp_df = pd.read_csv(meta_dir / "experimental_design.csv")\n\n    # 2. Process hourly weather\n    weather_csv = meta_dir / "weather_station_hourly.csv"\n    if weather_csv.exists():\n        weather_h = pd.read_csv(weather_csv)\n        weather_h["timestamp"] = pd.to_datetime(weather_h["timestamp"], utc=True)\n        weather_h["date"] = weather_h["timestamp"].dt.date\n\n        # Aggregate to daily\n        daily_list = []\n        for date_val, grp in weather_h.groupby("date"):\n            daily_list.append({\n                "date": pd.to_datetime(date_val),\n                "t_mean": grp["temperature"].mean(),\n                "t_max": grp["temperature"].max(),\n                "t_min": grp["temperature"].min(),\n                "rh_mean": grp["relative_humidity"].mean(),\n                "hours_above_90_rh": int((grp["relative_humidity"] >= 90.0).sum())\n            })\n        weather_d = pd.DataFrame(daily_list)\n    else:\n        weather_h = pd.DataFrame(columns=["timestamp", "temperature", "relative_humidity"])\n        weather_d = pd.DataFrame(columns=["date", "t_mean", "t_max", "t_min", "rh_mean"])\n\n    write_parquet(weather_h, processed_dir / "weather_hourly.parquet")\n    write_parquet(weather_d, processed_dir / "weather_daily.parquet")\n\n    # 3. Process time series mask files\n    image_records = []\n    first_frame_ref_area: dict[str, int] = {}\n\n    # Gather mask images sorted by timestamp per sample\n    sample_mask_paths: dict[str, list[Path]] = {}\n    for mask_path in sorted(ts_dir.rglob("*_mask.png")):\n        sample_id = mask_path.parent.name\n        sample_mask_paths.setdefault(sample_id, []).append(mask_path)\n\n    # First pass: Determine reference leaf area at frame 1 for each sample\n    for sample_id, mask_list in sample_mask_paths.items():\n        if not mask_list:\n            continue\n        first_mask_path = mask_list[0]\n        try:\n            arr = np.array(Image.open(first_mask_path))\n            m = compute_mask_metrics(arr)\n            ref_area = max(m["leaf_pixels_frame"], 1) # Avoid division by zero\n            first_frame_ref_area[sample_id] = ref_area\n        except Exception:\n            first_frame_ref_area[sample_id] = 10000\n\n    # Second pass: Compute per-image long metrics\n    sequence_records = []\n\n    for sample_id, mask_list in sample_mask_paths.items():\n        ref_area = first_frame_ref_area[sample_id]\n        plot_id = sample_id.split("_")[0]\n\n        # Match metadata plot details\n        plot_meta = exp_df[exp_df["plot_id"] == plot_id]\n        if not plot_meta.empty:\n            cultivar = plot_meta.iloc[0]["cultivar"]\n            treatment = plot_meta.iloc[0]["treatment"]\n            batch = plot_meta.iloc[0]["batch"]\n            year = plot_meta.iloc[0]["year"]\n        else:\n            cultivar, treatment, batch, year = "Unknown", "Untreated", 1, 2023\n\n        sample_images = []\n        first_timestamp = None\n\n        for mask_path in mask_list:\n            stem = mask_path.name.replace("_mask.png", "")\n            meta = parse_image_id(stem)\n            \n            if first_timestamp is None:\n                first_timestamp = meta["timestamp"]\n            hours_since_first = (meta["timestamp"] - first_timestamp).total_seconds() / 3600.0\n\n            arr = np.array(Image.open(mask_path))\n            m = compute_mask_metrics(arr)\n\n            # Primary severity formulation: diseased / first frame leaf area\n            severity = float(np.clip(m["diseased_pixels"] / float(ref_area), 0.0, 1.0))\n            \n            # Alternative metrics\n            per_frame_leaf = max(m["leaf_pixels_frame"], 1)\n            severity_per_frame = float(np.clip(m["diseased_pixels"] / float(per_frame_leaf), 0.0, 1.0))\n            necrosis_frac = float(m["necrosis_pixels"] / float(ref_area))\n            pustule_density = float(m["pustule_pixels"] / float(ref_area))\n\n            try:\n                rel_img_str = str(mask_path.relative_to(paths.project_root)).replace("_mask.png", ".jpg")\n            except ValueError:\n                rel_img_str = str(mask_path).replace("_mask.png", ".jpg")\n\n            rec = {\n                "image_id": meta["image_id"],\n                "sample_id": sample_id,\n                "plot_id": plot_id,\n                "leaf_no": meta["leaf_no"],\n                "cultivar": cultivar,\n                "treatment": treatment,\n                "batch": batch,\n                "year": year,\n                "timestamp": meta["timestamp"],\n                "hours_since_first": round(hours_since_first, 2),\n                "rel_image_path": rel_img_str,\n                "path": rel_img_str,\n                "severity": severity,\n                "severity_per_frame": severity_per_frame,\n                "necrosis_fraction": necrosis_frac,\n                "pustule_density": pustule_density,\n                "n_lesions": m["n_lesions"],\n                "mean_lesion_area": m["mean_lesion_area"],\n                "diseased_pixels": m["diseased_pixels"],\n                "reference_leaf_area": ref_area,\n                "missing_alignment": False,\n                "washed_off_markers": False,\n                "missing_masks": False\n            }\n            image_records.append(rec)\n            sample_images.append(rec)\n\n        if sample_images:\n            span_days = (sample_images[-1]["timestamp"] - sample_images[0]["timestamp"]).total_seconds() / 86400.0\n            sequence_records.append({\n                "sample_id": sample_id,\n                "n_images": len(sample_images),\n                "span_days": round(span_days, 2),\n                "cultivar": cultivar,\n                "plot_id": plot_id,\n                "treatment": treatment,\n                "batch": batch,\n                "year": year,\n                "dominant_disease": "Septoria / Rust"\n            })\n\n    images_df = pd.DataFrame(image_records)\n    sequences_df = pd.DataFrame(sequence_records)\n\n    write_parquet(images_df, processed_dir / "images_long.parquet")\n    write_parquet(sequences_df, processed_dir / "sequences.parquet")\n\n    logger.info(f"Exported {len(images_df)} images and {len(sequences_df)} sequences to {processed_dir}")\n    return {\n        "images_count": len(images_df),\n        "sequences_count": len(sequences_df),\n        "weather_days": len(weather_d)\n    }\n\ndef main() -> int:\n    process_eth_dataset()\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/features/storage_estimate.py').write_text('"""Storage Estimation Tool for Cached Vision Backbone Features.\n\nCalculates estimated disk space requirements for cached DINOv2 feature embeddings\n(CLS token, mean patch token, and full patch token grids) across different image\nresolutions and backbone architectures before starting feature extraction.\n"""\n\nimport sys\nimport argparse\nfrom pathlib import Path\nfrom typing import Dict, Any, List, Optional\nimport pandas as pd\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\n\nlogger = get_logger(__name__)\n\n# Backbone embedding dimension mapping\nBACKBONE_DIMENSIONS = {\n    "small": 384,\n    "dinov2-small": 384,\n    "facebook/dinov2-small": 384,\n    "base": 768,\n    "dinov2-base": 768,\n    "facebook/dinov2-base": 768,\n}\n\ndef calculate_image_feature_size(\n    resolution: int = 448,\n    patch_size: int = 14,\n    embed_dim: int = 384,\n    dtype_bytes: int = 2  # float16 = 2 bytes\n) -> Dict[str, Any]:\n    """Calculate exact storage footprint per image in bytes and MB.\n\n    Args:\n        resolution: Square input resolution (H = W = resolution).\n        patch_size: Patch size in pixels (default 14 for DINOv2).\n        embed_dim: Feature embedding dimension (384 for Small, 768 for Base).\n        dtype_bytes: Bytes per float value (2 bytes for float16, 4 for float32).\n\n    Returns:\n        Dict with grid dimensions, tensor shapes, and byte counts.\n    """\n    grid_h = resolution // patch_size\n    grid_w = resolution // patch_size\n    n_patches = grid_h * grid_w\n\n    cls_bytes = embed_dim * dtype_bytes\n    mean_patch_bytes = embed_dim * dtype_bytes\n    grid_bytes = n_patches * embed_dim * dtype_bytes\n    metadata_overhead = 500  # Index entry and safetensors header overhead\n\n    total_bytes = cls_bytes + mean_patch_bytes + grid_bytes + metadata_overhead\n    total_mb = total_bytes / (1024 * 1024)\n\n    return {\n        "resolution": resolution,\n        "patch_size": patch_size,\n        "grid_shape": (grid_h, grid_w),\n        "n_patches": n_patches,\n        "embed_dim": embed_dim,\n        "cls_bytes": cls_bytes,\n        "mean_patch_bytes": mean_patch_bytes,\n        "grid_bytes": grid_bytes,\n        "total_bytes_per_image": total_bytes,\n        "mb_per_image": total_mb\n    }\n\ndef estimate_storage_for_dataset(\n    n_images: int,\n    backbone_name: str = "small",\n    resolution: int = 448,\n    patch_size: int = 14,\n    dtype_bytes: int = 2\n) -> Dict[str, Any]:\n    """Estimate total storage for a given dataset image count.\n\n    Args:\n        n_images: Total number of images in dataset.\n        backbone_name: Model variant ("small" or "base").\n        resolution: Image resolution (e.g., 448, 672, 896).\n        patch_size: Patch size (default 14).\n        dtype_bytes: Data type byte size (default 2 for fp16).\n\n    Returns:\n        Dict with total MB and GB required.\n    """\n    embed_dim = BACKBONE_DIMENSIONS.get(backbone_name.lower(), 384)\n    per_image = calculate_image_feature_size(\n        resolution=resolution,\n        patch_size=patch_size,\n        embed_dim=embed_dim,\n        dtype_bytes=dtype_bytes\n    )\n    total_bytes = per_image["total_bytes_per_image"] * n_images\n    total_mb = total_bytes / (1024 * 1024)\n    total_gb = total_bytes / (1024 * 1024 * 1024)\n\n    return {\n        "n_images": n_images,\n        "backbone": backbone_name,\n        "resolution": resolution,\n        "embed_dim": embed_dim,\n        "mb_per_image": round(per_image["mb_per_image"], 4),\n        "total_mb": round(total_mb, 2),\n        "total_gb": round(total_gb, 4)\n    }\n\ndef generate_storage_report(\n    static_n: int = 1000,\n    eth_n: int = 12520,\n    available_gb: float = 15.0\n) -> Dict[str, Any]:\n    """Generate comprehensive storage matrix across resolutions and backbones.\n\n    Args:\n        static_n: Number of static in-the-wild images.\n        eth_n: Number of ETH Zurich wheat images.\n        available_gb: Free storage limit (default 15 GB for Google Drive free tier).\n\n    Returns:\n        Dict containing summary DataFrame, warning flags, and recommendations.\n    """\n    records = []\n    backbones = ["small", "base"]\n    resolutions = [448, 672, 896]\n    total_images = static_n + eth_n\n\n    for bb in backbones:\n        for res in resolutions:\n            est_static = estimate_storage_for_dataset(static_n, bb, res)\n            est_eth = estimate_storage_for_dataset(eth_n, bb, res)\n            est_total = estimate_storage_for_dataset(total_images, bb, res)\n\n            records.append({\n                "backbone": bb.capitalize(),\n                "resolution": f"{res}x{res}",\n                "grid_shape": f"{res//14}x{res//14}",\n                "mb_per_img": est_total["mb_per_image"],\n                "static_gb": est_static["total_gb"],\n                "eth_gb": est_eth["total_gb"],\n                "total_gb": est_total["total_gb"],\n                "fits_drive_15gb": est_total["total_gb"] <= available_gb\n            })\n\n    df = pd.DataFrame(records)\n    warnings = []\n    exceeds_count = sum(~df["fits_drive_15gb"])\n    if exceeds_count > 0:\n        warnings.append(\n            f"WARNING: {exceeds_count} feature configurations exceed free Google Drive capacity ({available_gb} GB)."\n        )\n\n    recommendations = [\n        "1. DEFAULT RECOMMENDATION: Use DINOv2-Small at 448x448 resolution (~10.5 GB total for all 13,520 images).",\n        "2. STORAGE-CONSTRAINED OPTION: Extract patch tokens only for training splits, or extract 448x448 first.",\n        "3. HIGH-RESOLUTION OPTION: DINOv2-Base at 896x896 requires ~94 GB; run ETH on downscaled crops or extract on demand."\n    ]\n\n    return {\n        "static_image_count": static_n,\n        "eth_image_count": eth_n,\n        "total_images": total_images,\n        "available_drive_gb": available_gb,\n        "storage_table": df,\n        "warnings": warnings,\n        "recommendations": recommendations\n    }\n\ndef print_storage_summary(static_n: int = 1000, eth_n: int = 12520, available_gb: float = 15.0) -> None:\n    """Print formatted storage estimate report to stdout."""\n    report = generate_storage_report(static_n, eth_n, available_gb)\n    print("=" * 70)\n    print("FEATURE CACHE STORAGE FOOTPRINT ESTIMATOR")\n    print("=" * 70)\n    print(f"Static Images: {report[\'static_image_count\']:,} | ETH Images: {report[\'eth_image_count\']:,} | Total: {report[\'total_images\']:,}")\n    print(f"Target Storage Cap: {report[\'available_drive_gb\']} GB")\n    print("-" * 70)\n    df = report["storage_table"]\n    print(df.to_string(index=False))\n    print("-" * 70)\n    for w in report["warnings"]:\n        print(f"[WARNING] {w}")\n    print("\\nRECOMMENDATIONS:")\n    for r in report["recommendations"]:\n        print(f"  {r}")\n    print("=" * 70)\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="Estimate disk footprint for feature caching.")\n    parser.add_argument("--static-n", type=int, default=1000, help="Number of static images.")\n    parser.add_argument("--eth-n", type=int, default=12520, help="Number of ETH images.")\n    parser.add_argument("--available-gb", type=float, default=15.0, help="Drive limit in GB.")\n    args = parser.parse_args()\n\n    print_storage_summary(static_n=args.static_n, eth_n=args.eth_n, available_gb=args.available_gb)\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/features/extract_features.py').write_text('"""Feature Extractor and Caching Engine for Vision Backbones (DINOv2).\n\nExtracts CLS tokens, mean patch tokens, and patch token grids from frozen DINOv2 vision\nbackbones, supports ETH Zurich macro image cropping and tiling, and caches representations\nto sharded .safetensors files with cryptographic cache-key verification and resumability.\n"""\n\nimport sys\nimport os\nimport json\nimport hashlib\nimport argparse\nfrom pathlib import Path\nfrom typing import Dict, Any, List, Tuple, Optional, Union\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom safetensors.torch import save_file, load_file\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\nfrom ml.utils.io import read_parquet\n\nlogger = get_logger(__name__)\n\n# Standard ImageNet normalization constants\nIMAGENET_MEAN = [0.485, 0.456, 0.406]\nIMAGENET_STD = [0.229, 0.224, 0.225]\n\nclass StubBackbone(nn.Module):\n    """Lightweight stub vision backbone for CPU unit testing without network/GPU dependencies."""\n    def __init__(self, embed_dim: int = 384, patch_size: int = 14):\n        super().__init__()\n        self.embed_dim = embed_dim\n        self.patch_size = patch_size\n        self.proj = nn.Conv2d(3, embed_dim, kernel_size=patch_size, stride=patch_size)\n\n    def forward(self, x: torch.Tensor) -> Dict[str, torch.Tensor]:\n        # x: (B, 3, H, W)\n        b, c, h, w = x.shape\n        gh, gw = h // self.patch_size, w // self.patch_size\n        feat_map = self.proj(x)  # (B, D, gh, gw)\n        patches = feat_map.flatten(2).transpose(1, 2)  # (B, gh*gw, D)\n        cls_token = patches.mean(dim=1)  # (B, D)\n        tokens = torch.cat([cls_token.unsqueeze(1), patches], dim=1)  # (B, 1 + gh*gw, D)\n        \n        class StubOutput:\n            def __init__(self, last_hidden_state):\n                self.last_hidden_state = last_hidden_state\n        return StubOutput(tokens)\n\ndef compute_cache_key(\n    backbone_id: str,\n    revision: str,\n    resolution: int,\n    patch_size: int,\n    eth_mode: str = "downscaled_crop"\n) -> str:\n    """Compute deterministic SHA-256 hash key for cache verification.\n\n    Args:\n        backbone_id: Hugging Face model identifier (e.g. "facebook/dinov2-small").\n        revision: Git commit hash or revision tag (e.g. "main").\n        resolution: Target spatial resolution.\n        patch_size: Model patch resolution.\n        eth_mode: ETH macro processing mode ("downscaled_crop" or "tiled").\n\n    Returns:\n        16-character hexadecimal hash string.\n    """\n    raw_str = f"{backbone_id}::{revision}::{resolution}::{patch_size}::{eth_mode}"\n    return hashlib.sha256(raw_str.encode("utf-8")).hexdigest()[:16]\n\ndef preprocess_image(\n    image: Image.Image,\n    target_resolution: int = 448,\n    patch_size: int = 14,\n    mean: List[float] = IMAGENET_MEAN,\n    std: List[float] = IMAGENET_STD\n) -> Tuple[torch.Tensor, Dict[str, Any]]:\n    """Resize image preserving aspect ratio and pad to exact multiples of patch size.\n\n    Args:\n        image: PIL Image instance (RGB).\n        target_resolution: Target max side length.\n        patch_size: Patch stride in pixels (default 14).\n        mean: Normalization channel means.\n        std: Normalization channel standard deviations.\n\n    Returns:\n        Tuple of (normalized tensor (3, padded_h, padded_w), metadata_dict).\n    """\n    orig_w, orig_h = image.size\n    \n    # Calculate scale factor to fit within target_resolution\n    scale = min(target_resolution / orig_w, target_resolution / orig_h)\n    new_w = max(1, int(round(orig_w * scale)))\n    new_h = max(1, int(round(orig_h * scale)))\n    \n    # Resize preserving aspect ratio\n    resized_img = image.resize((new_w, new_h), Image.Resampling.BILINEAR)\n    \n    # Calculate padded size (nearest upper multiple of patch_size)\n    padded_w = int(np.ceil(new_w / patch_size) * patch_size)\n    padded_h = int(np.ceil(new_h / patch_size) * patch_size)\n    \n    # Convert to NumPy array and pad with edge pixels or zero\n    img_np = np.array(resized_img.convert("RGB"), dtype=np.float32) / 255.0\n    \n    # Pad array\n    pad_h = padded_h - new_h\n    pad_w = padded_w - new_w\n    padded_np = np.pad(img_np, ((0, pad_h), (0, pad_w), (0, 0)), mode="edge")\n    \n    # Normalize with mean and std\n    mean_arr = np.array(mean, dtype=np.float32).reshape(1, 1, 3)\n    std_arr = np.array(std, dtype=np.float32).reshape(1, 1, 3)\n    norm_np = (padded_np - mean_arr) / std_arr\n    \n    # Convert to C x H x W torch Tensor\n    tensor = torch.from_numpy(norm_np.transpose(2, 0, 1)).float()\n    \n    meta = {\n        "original_size": (orig_h, orig_w),\n        "resized_size": (new_h, new_w),\n        "padded_size": (padded_h, padded_w),\n        "grid_shape": (padded_h // patch_size, padded_w // patch_size)\n    }\n    return tensor, meta\n\ndef extract_image_tiles(\n    image: Image.Image,\n    tile_size: int = 448,\n    tile_overlap: int = 56,\n    patch_size: int = 14\n) -> Tuple[List[torch.Tensor], List[Tuple[int, int, int, int]], List[Dict[str, Any]]]:\n    """Extract overlapping sliding-window tiles from high-resolution macro photos.\n\n    Args:\n        image: High-resolution PIL RGB image.\n        tile_size: Square tile dimension.\n        tile_overlap: Pixel overlap between adjacent tiles.\n        patch_size: Model patch size.\n\n    Returns:\n        Tuple of (list of tile tensors, list of (x1, y1, x2, y2) bounds, list of tile metadata).\n    """\n    w, h = image.size\n    stride = tile_size - tile_overlap\n    \n    x_steps = list(range(0, max(1, w - tile_size + 1), stride))\n    if x_steps[-1] + tile_size < w:\n        x_steps.append(w - tile_size)\n        \n    y_steps = list(range(0, max(1, h - tile_size + 1), stride))\n    if y_steps[-1] + tile_size < h:\n        y_steps.append(h - tile_size)\n        \n    tile_tensors = []\n    tile_coords = []\n    tile_metas = []\n    \n    for y1 in y_steps:\n        for x1 in x_steps:\n            x2 = min(w, x1 + tile_size)\n            y2 = min(h, y1 + tile_size)\n            crop_img = image.crop((x1, y1, x2, y2))\n            t_tensor, t_meta = preprocess_image(crop_img, target_resolution=tile_size, patch_size=patch_size)\n            tile_tensors.append(t_tensor)\n            tile_coords.append((x1, y1, x2, y2))\n            tile_metas.append(t_meta)\n            \n    return tile_tensors, tile_coords, tile_metas\n\nclass FeatureExtractorEngine:\n    """Core feature extraction runner supporting HF DINOv2 and Stub backbones."""\n    def __init__(\n        self,\n        backbone_id: str = "facebook/dinov2-small",\n        revision: str = "main",\n        patch_size: int = 14,\n        is_stub: bool = False,\n        device: Optional[str] = None\n    ):\n        self.backbone_id = backbone_id\n        self.revision = revision\n        self.patch_size = patch_size\n        self.is_stub = is_stub\n        \n        if device is None:\n            self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n        else:\n            self.device = torch.device(device)\n            \n        if is_stub:\n            embed_dim = 384 if "small" in backbone_id else 768\n            self.model = StubBackbone(embed_dim=embed_dim, patch_size=patch_size)\n            self.embed_dim = embed_dim\n        else:\n            try:\n                from transformers import AutoModel\n                logger.info(f"Loading HF backbone model: {backbone_id} (revision: {revision})...")\n                self.model = AutoModel.from_pretrained(backbone_id, revision=revision)\n                self.embed_dim = getattr(self.model.config, "hidden_size", 384)\n            except Exception as e:\n                logger.warning(f"Failed to load HF backbone ({e}). Falling back to StubBackbone for CPU.")\n                embed_dim = 384 if "small" in backbone_id else 768\n                self.model = StubBackbone(embed_dim=embed_dim, patch_size=patch_size)\n                self.embed_dim = embed_dim\n                self.is_stub = True\n                \n        self.model.to(self.device)\n        self.model.eval()\n        for p in self.model.parameters():\n            p.requires_grad = False\n\n    @torch.no_grad()\n    def extract_features(\n        self,\n        input_tensor: torch.Tensor,\n        use_amp: bool = True\n    ) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:\n        """Extract CLS token, mean patch token, and patch token grid.\n\n        Args:\n            input_tensor: Single image tensor (3, H, W) or batch tensor (B, 3, H, W).\n            use_amp: Whether to use mixed precision float16 on GPU.\n\n        Returns:\n            Tuple of (cls_token [B, D], mean_patch_token [B, D], patch_grid [B, gh, gw, D]).\n        """\n        if input_tensor.ndim == 3:\n            input_tensor = input_tensor.unsqueeze(0)\n            \n        b, c, h, w = input_tensor.shape\n        gh, gw = h // self.patch_size, w // self.patch_size\n        \n        input_tensor = input_tensor.to(self.device)\n        amp_device = "cuda" if self.device.type == "cuda" else "cpu"\n        \n        with torch.amp.autocast(device_type=amp_device, enabled=use_amp and self.device.type == "cuda"):\n            outputs = self.model(input_tensor)\n            \n        last_hidden = outputs.last_hidden_state  # (B, 1 + gh*gw, D)\n        cls_token = last_hidden[:, 0, :]  # (B, D)\n        patch_tokens = last_hidden[:, 1:, :]  # (B, gh*gw, D)\n        mean_patch_token = patch_tokens.mean(dim=1)  # (B, D)\n        patch_grid = patch_tokens.reshape(b, gh, gw, self.embed_dim)  # (B, gh, gw, D)\n        \n        return cls_token.cpu().half(), mean_patch_token.cpu().half(), patch_grid.cpu().half()\n\ndef run_feature_extraction_pipeline(\n    manifest_df: pd.DataFrame,\n    output_dir: Optional[Path] = None,\n    backbone_id: str = "facebook/dinov2-small",\n    revision: str = "main",\n    resolution: int = 448,\n    patch_size: int = 14,\n    shard_size: int = 500,\n    eth_mode: str = "downscaled_crop",\n    is_stub: bool = False,\n    use_gpu: bool = False\n) -> Dict[str, Any]:\n    """Run full feature extraction pipeline over manifest images, saving sharded safetensors.\n\n    Args:\n        manifest_df: DataFrame containing image_id and rel_image_path columns.\n        output_dir: Target output cache directory.\n        backbone_id: DINOv2 model ID.\n        revision: Model revision tag.\n        resolution: Target spatial resolution.\n        patch_size: Patch stride size.\n        shard_size: Number of images per safetensors file.\n        eth_mode: ETH macro processing mode.\n        is_stub: Force stub model for testing.\n        use_gpu: Use GPU if available.\n\n    Returns:\n        Summary dict containing cache_key, total_images, processed_images, and throughput stats.\n    """\n    paths = get_paths()\n    cache_key = compute_cache_key(backbone_id, revision, resolution, patch_size, eth_mode)\n    \n    if output_dir is None:\n        cache_dir = paths.data_root / "processed" / "features" / cache_key\n    else:\n        if (output_dir / "index.json").exists() or output_dir.name == cache_key:\n            cache_dir = output_dir\n        else:\n            cache_dir = output_dir / cache_key\n        \n    cache_dir.mkdir(parents=True, exist_ok=True)\n    index_path = cache_dir / "index.json"\n    \n    # Check existing index for resumption & cache key verification\n    index_data = {\n        "cache_key": cache_key,\n        "backbone_id": backbone_id,\n        "revision": revision,\n        "resolution": resolution,\n        "patch_size": patch_size,\n        "eth_mode": eth_mode,\n        "images": {}\n    }\n    \n    if index_path.exists():\n        with open(index_path, "r", encoding="utf-8") as f:\n            existing = json.load(f)\n        if existing.get("cache_key") != cache_key:\n            raise ValueError(\n                f"Cache key mismatch! Existing cache in {cache_dir} has key \'{existing.get(\'cache_key\')}\', "\n                f"but current extraction settings produced \'{cache_key}\'. Refusing to mix caches."\n            )\n        index_data = existing\n        logger.info(f"Resuming feature extraction. Found {len(index_data[\'images\'])} previously extracted images.")\n\n    device_str = "cuda" if use_gpu and torch.cuda.is_available() else "cpu"\n    engine = FeatureExtractorEngine(\n        backbone_id=backbone_id,\n        revision=revision,\n        patch_size=patch_size,\n        is_stub=is_stub,\n        device=device_str\n    )\n    \n    processed_count = 0\n    skipped_count = 0\n    current_shard_tensors = {}\n    current_shard_idx = len(list(cache_dir.glob("shard_*.safetensors")))\n    \n    records = manifest_df.to_dict("records") if not manifest_df.empty else []\n    \n    for row in records:\n        img_id = row.get("image_id")\n        if not img_id:\n            continue\n            \n        # Check if already processed (resumable)\n        if img_id in index_data["images"]:\n            skipped_count += 1\n            continue\n            \n        rel_path = row.get("rel_image_path") or row.get("path") or row.get("rel_mask_path")\n        if not rel_path:\n            continue\n            \n        full_img_path = paths.project_root / rel_path\n        if not full_img_path.exists():\n            # Fall back to checking under data_root/raw\n            full_img_path = paths.raw_data / rel_path\n            if not full_img_path.exists():\n                mask_rel = row.get("rel_mask_path") or rel_path.replace(".jpg", "_mask.png")\n                full_img_path = paths.project_root / mask_rel\n                if not full_img_path.exists():\n                    full_img_path = paths.raw_data / mask_rel\n                    if not full_img_path.exists():\n                        logger.warning(f"Image path missing: {rel_path}. Skipping.")\n                        continue\n                \n        try:\n            with Image.open(full_img_path) as img:\n                img_rgb = img.convert("RGB")\n                tensor, meta = preprocess_image(img_rgb, target_resolution=resolution, patch_size=patch_size)\n                cls_t, mean_t, patch_t = engine.extract_features(tensor, use_amp=(device_str == "cuda"))\n                \n            shard_name = f"shard_{current_shard_idx:04d}.safetensors"\n            current_shard_tensors[f"{img_id}_cls"] = cls_t.squeeze(0)\n            current_shard_tensors[f"{img_id}_mean"] = mean_t.squeeze(0)\n            current_shard_tensors[f"{img_id}_patch"] = patch_t.squeeze(0)\n            \n            index_data["images"][img_id] = {\n                "shard": shard_name,\n                "original_size": list(meta["original_size"]),\n                "resized_size": list(meta["resized_size"]),\n                "padded_size": list(meta["padded_size"]),\n                "grid_shape": list(meta["grid_shape"])\n            }\n            processed_count += 1\n            \n            # Save shard when capacity reached\n            if len(current_shard_tensors) >= (shard_size * 3):\n                shard_file = cache_dir / shard_name\n                save_file(current_shard_tensors, shard_file)\n                current_shard_tensors = {}\n                current_shard_idx += 1\n                \n                # Periodically flush index JSON\n                with open(index_path, "w", encoding="utf-8") as f:\n                    json.dump(index_data, f, indent=2)\n                    \n        except Exception as e:\n            logger.warning(f"Error processing image {img_id}: {e}")\n            continue\n\n    # Flush remaining tensors in last shard\n    if current_shard_tensors:\n        shard_name = f"shard_{current_shard_idx:04d}.safetensors"\n        shard_file = cache_dir / shard_name\n        save_file(current_shard_tensors, shard_file)\n\n    with open(index_path, "w", encoding="utf-8") as f:\n        json.dump(index_data, f, indent=2)\n        \n    logger.info(f"Feature extraction completed. Processed: {processed_count}, Skipped: {skipped_count}, Total Indexed: {len(index_data[\'images\'])}.")\n    \n    return {\n        "cache_key": cache_key,\n        "cache_dir": str(cache_dir),\n        "total_manifest_images": len(records),\n        "processed_images": processed_count,\n        "skipped_images": skipped_count,\n        "total_indexed_images": len(index_data["images"]),\n        "num_shards": current_shard_idx + (1 if current_shard_tensors else 0)\n    }\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="Extract and cache DINOv2 vision features.")\n    parser.add_argument("--manifest", type=str, default=None, help="Path to manifest parquet.")\n    parser.add_argument("--output-dir", type=str, default=None, help="Cache directory.")\n    parser.add_argument("--backbone", type=str, default="facebook/dinov2-small", help="Model ID.")\n    parser.add_argument("--resolution", type=int, default=448, help="Target resolution.")\n    parser.add_argument("--stub", action="store_true", help="Force stub backbone for CPU testing.")\n    parser.add_argument("--gpu", action="store_true", help="Use GPU acceleration.")\n    args = parser.parse_args()\n\n    paths = get_paths()\n    if args.manifest:\n        m_path = Path(args.manifest)\n    else:\n        m_path = paths.project_root / "data" / "processed" / "static" / "manifest.parquet"\n        \n    if m_path.exists():\n        df = read_parquet(m_path)\n    else:\n        logger.warning(f"Manifest file {m_path} not found. Running with empty DataFrame.")\n        df = pd.DataFrame()\n        \n    out_dir = Path(args.output_dir) if args.output_dir else None\n    res = run_feature_extraction_pipeline(\n        manifest_df=df,\n        output_dir=out_dir,\n        backbone_id=args.backbone,\n        resolution=args.resolution,\n        is_stub=args.stub,\n        use_gpu=args.gpu\n    )\n    print("Extraction Summary:", res)\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/features/ood_stats.py').write_text('"""Out-Of-Distribution (OOD) Feature Statistics Calculator.\n\nComputes per-source mean vectors, shrinkage-regularized covariance matrices, precision matrices,\nand 95th/99th percentile Mahalanobis distance thresholds on validation feature embeddings for Phase 8 safety layer.\n"""\n\nimport sys\nimport json\nimport argparse\nfrom pathlib import Path\nfrom typing import Dict, Any, List, Tuple, Optional\nimport numpy as np\nimport torch\nfrom safetensors.torch import save_file, load_file\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\n\nlogger = get_logger(__name__)\n\ndef compute_mahalanobis_distance(\n    x: np.ndarray,\n    mean_vec: np.ndarray,\n    precision_matrix: np.ndarray\n) -> np.ndarray:\n    """Compute Mahalanobis distance for a set of feature vectors.\n\n    D_M(x) = sqrt((x - mu)^T * Precision * (x - mu))\n\n    Args:\n        x: Feature matrix of shape (N, D) or single vector of shape (D,).\n        mean_vec: Mean feature vector of shape (D,).\n        precision_matrix: Inverse covariance (precision) matrix of shape (D, D).\n\n    Returns:\n        NumPy array of scalar distances of shape (N,).\n    """\n    if x.ndim == 1:\n        x = x.reshape(1, -1)\n        \n    diff = x - mean_vec.reshape(1, -1)  # (N, D)\n    # (N, D) @ (D, D) -> (N, D)\n    left = np.dot(diff, precision_matrix)\n    # Row-wise dot product\n    dist_sq = np.sum(left * diff, axis=1)\n    # Clip numerical underflow\n    dist_sq = np.maximum(dist_sq, 0.0)\n    return np.sqrt(dist_sq)\n\ndef compute_ood_statistics(\n    train_features: np.ndarray,\n    val_features: Optional[np.ndarray] = None,\n    gamma: float = 0.01,\n    percentiles: List[float] = [95.0, 99.0]\n) -> Dict[str, Any]:\n    """Compute shrinkage-regularized covariance, precision, and validation percentiles.\n\n    Args:\n        train_features: Training set CLS embeddings of shape (N_train, D).\n        val_features: Optional validation set CLS embeddings of shape (N_val, D).\n        gamma: Shrinkage regularization intensity (0 < gamma <= 1).\n        percentiles: Percentile thresholds to compute on validation set.\n\n    Returns:\n        Dict containing mean vector, precision matrix, and calculated percentile values.\n    """\n    if train_features.size == 0:\n        raise ValueError("Cannot compute OOD stats on empty training feature array.")\n        \n    if train_features.ndim == 1:\n        train_features = train_features.reshape(1, -1)\n        \n    n_train, embed_dim = train_features.shape\n    \n    # Compute mean vector\n    mean_vec = np.mean(train_features, axis=0)  # (D,)\n    \n    # Compute sample covariance matrix\n    if n_train > 1:\n        cov_sample = np.cov(train_features, rowvar=False)  # (D, D)\n    else:\n        cov_sample = np.eye(embed_dim)\n        \n    # Shrinkage regularization: Sigma_shrunk = (1 - gamma) * S + gamma * nu * I\n    nu = np.trace(cov_sample) / float(embed_dim) if embed_dim > 0 else 1.0\n    cov_shrunk = (1.0 - gamma) * cov_sample + gamma * nu * np.eye(embed_dim)\n    \n    # Precision matrix (inverse covariance)\n    try:\n        precision_mat = np.linalg.inv(cov_shrunk)\n    except np.linalg.LinAlgError:\n        logger.warning("Covariance matrix inversion failed. Falling back to pseudo-inverse.")\n        precision_mat = np.linalg.pinv(cov_shrunk)\n\n    # Compute validation Mahalanobis distances\n    if val_features is not None and val_features.size > 0:\n        if val_features.ndim == 1:\n            val_features = val_features.reshape(1, -1)\n        val_dists = compute_mahalanobis_distance(val_features, mean_vec, precision_mat)\n    else:\n        # Fall back to self-distances on training set if validation is absent\n        val_dists = compute_mahalanobis_distance(train_features, mean_vec, precision_mat)\n\n    pct_dict = {}\n    for p in percentiles:\n        val_p = float(np.percentile(val_dists, p))\n        pct_dict[f"p{int(p)}"] = round(val_p, 4)\n        \n    return {\n        "mean_vector": mean_vec,\n        "precision_matrix": precision_mat,\n        "embed_dim": embed_dim,\n        "n_train_samples": n_train,\n        "n_val_samples": len(val_dists),\n        "gamma": gamma,\n        "percentiles": pct_dict,\n        "sample_val_distances": val_dists[:10].tolist()\n    }\n\ndef process_ood_stats(\n    cache_dir: Path,\n    output_dir: Optional[Path] = None,\n    gamma: float = 0.01,\n    percentiles: List[float] = [95.0, 99.0]\n) -> Dict[str, Any]:\n    """Load cached features, compute OOD statistics, and export to safetensors & JSON.\n\n    Args:\n        cache_dir: Feature cache directory containing index.json and shard files.\n        output_dir: Destination folder (defaults to cache_dir).\n        gamma: Covariance shrinkage factor.\n        percentiles: Validation percentile thresholds.\n\n    Returns:\n        Metadata dictionary summarizing OOD calculation.\n    """\n    if output_dir is None:\n        output_dir = cache_dir\n    output_dir.mkdir(parents=True, exist_ok=True)\n    \n    index_path = cache_dir / "index.json"\n    if not index_path.exists():\n        raise FileNotFoundError(f"Feature cache index not found at {index_path}.")\n        \n    with open(index_path, "r", encoding="utf-8") as f:\n        index_data = json.load(f)\n\n    image_entries = index_data.get("images", {})\n    if not image_entries:\n        logger.warning(f"No image entries found in index {index_path}. Returning dummy OOD stats.")\n        # Generate stub stats for testing\n        stub_train = np.random.randn(20, 384).astype(np.float32)\n        stub_val = np.random.randn(10, 384).astype(np.float32)\n        stats = compute_ood_statistics(stub_train, stub_val, gamma=gamma, percentiles=percentiles)\n    else:\n        # Group entries by shard file\n        shards = {}\n        for img_id, entry in image_entries.items():\n            shards.setdefault(entry["shard"], []).append(img_id)\n            \n        cls_vectors = []\n        for shard_name, img_ids in shards.items():\n            shard_path = cache_dir / shard_name\n            if not shard_path.exists():\n                continue\n            tensors = load_file(shard_path)\n            for img_id in img_ids:\n                key = f"{img_id}_cls"\n                if key in tensors:\n                    cls_vectors.append(tensors[key].cpu().float().numpy())\n                    \n        if not cls_vectors:\n            raise ValueError(f"No CLS feature vectors found in shard files in {cache_dir}.")\n            \n        cls_matrix = np.stack(cls_vectors, axis=0)  # (N, D)\n        \n        # 80/20 train/val split for OOD percentile estimation\n        n = len(cls_matrix)\n        split_idx = max(1, int(n * 0.8))\n        train_cls = cls_matrix[:split_idx]\n        val_cls = cls_matrix[split_idx:] if split_idx < n else cls_matrix\n        \n        stats = compute_ood_statistics(train_cls, val_cls, gamma=gamma, percentiles=percentiles)\n\n    # Export tensors to safetensors format\n    tensors_to_save = {\n        "ood_mean": torch.from_numpy(stats["mean_vector"]).float(),\n        "ood_precision": torch.from_numpy(stats["precision_matrix"]).float()\n    }\n    safetensors_path = output_dir / "ood_stats.safetensors"\n    save_file(tensors_to_save, safetensors_path)\n    \n    # Export metadata to JSON format\n    json_metadata = {\n        "embed_dim": stats["embed_dim"],\n        "n_train_samples": stats["n_train_samples"],\n        "n_val_samples": stats["n_val_samples"],\n        "gamma": stats["gamma"],\n        "percentiles": stats["percentiles"],\n        "safetensors_file": "ood_stats.safetensors"\n    }\n    json_path = output_dir / "ood_stats.json"\n    with open(json_path, "w", encoding="utf-8") as f:\n        json.dump(json_metadata, f, indent=2)\n        \n    logger.info(f"OOD Statistics exported to {safetensors_path} and {json_path}.")\n    return json_metadata\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description="Calculate OOD feature statistics from cached features.")\n    parser.add_argument("--cache-dir", type=str, required=True, help="Directory containing feature cache.")\n    parser.add_argument("--output-dir", type=str, default=None, help="Output destination folder.")\n    parser.add_argument("--gamma", type=float, default=0.01, help="Shrinkage factor.")\n    args = parser.parse_args()\n\n    c_dir = Path(args.cache_dir)\n    o_dir = Path(args.output_dir) if args.output_dir else None\n    res = process_ood_stats(c_dir, output_dir=o_dir, gamma=args.gamma)\n    print("OOD Stats Metadata:", res)\n    return 0\n\nif __name__ == "__main__":\n    sys.exit(main())\n')
    Path('/content/ml/features/feature_dataset.py').write_text('"""PyTorch Dataset Class for Cached Vision Backbone Features.\n\nProvides lazy, memory-efficient loading of cached DINOv2 feature embeddings (.safetensors shards)\nalong with ground truth masks (resized to patch grid spatial dimensions) for downstream head training.\n"""\n\nimport sys\nimport json\nfrom pathlib import Path\nfrom typing import Dict, Any, List, Tuple, Optional, Union\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport torch\nfrom torch.utils.data import Dataset\nfrom safetensors.torch import load_file\n\n# Ensure project root is in sys.path\nproject_root = Path(__file__).resolve().parent.parent.parent\nif str(project_root) not in sys.path:\n    sys.path.insert(0, str(project_root))\n\nfrom ml.utils.paths import get_paths\nfrom ml.utils.logging_utils import get_logger\n\nlogger = get_logger(__name__)\n\nclass CachedFeatureDataset(Dataset):\n    """Lazy-loading PyTorch Dataset reading sharded .safetensors feature embeddings."""\n\n    def __init__(\n        self,\n        cache_dir: Union[str, Path],\n        manifest_df: Optional[pd.DataFrame] = None,\n        split: Optional[str] = None,\n        transform: Optional[Any] = None\n    ):\n        """Initialize dataset from feature cache directory and optional manifest filters.\n\n        Args:\n            cache_dir: Path to directory containing index.json and shard safetensors.\n            manifest_df: Optional manifest DataFrame to filter image IDs or load mask paths.\n            split: Optional split filter string ("train", "val", "test", "calibration").\n            transform: Optional feature transform function.\n        """\n        self.cache_dir = Path(cache_dir)\n        self.index_path = self.cache_dir / "index.json"\n        \n        if not self.index_path.exists():\n            raise FileNotFoundError(f"Feature cache index.json missing at {self.index_path}")\n            \n        with open(self.index_path, "r", encoding="utf-8") as f:\n            self.index_data = json.load(f)\n\n        self.cache_key = self.index_data.get("cache_key", "")\n        self.backbone_id = self.index_data.get("backbone_id", "")\n        self.embed_dim = self.index_data.get("embed_dim", 384)\n        self.patch_size = self.index_data.get("patch_size", 14)\n        self.transform = transform\n        \n        all_images = self.index_data.get("images", {})\n        \n        # Merge with manifest DataFrame if provided\n        self.items = []\n        paths = get_paths()\n        \n        if manifest_df is not None and not manifest_df.empty:\n            df = manifest_df.copy()\n            if split is not None and "split" in df.columns:\n                df = df[df["split"] == split]\n                \n            manifest_map = {row["image_id"]: row for row in df.to_dict("records") if "image_id" in row}\n            \n            for img_id, entry in all_images.items():\n                if img_id in manifest_map:\n                    m_row = manifest_map[img_id]\n                    entry_copy = dict(entry)\n                    entry_copy["image_id"] = img_id\n                    entry_copy["rel_image_path"] = m_row.get("rel_image_path")\n                    entry_copy["rel_lesion_mask_path"] = m_row.get("rel_lesion_mask_path")\n                    entry_copy["rel_leaf_mask_path"] = m_row.get("rel_leaf_mask_path")\n                    entry_copy["split"] = m_row.get("split", "train")\n                    entry_copy["source"] = m_row.get("source", "unknown")\n                    entry_copy["severity"] = m_row.get("severity", 0.0)\n                    entry_copy["cluster_id"] = m_row.get("cluster_id", 0)\n                    self.items.append(entry_copy)\n        else:\n            for img_id, entry in all_images.items():\n                entry_copy = dict(entry)\n                entry_copy["image_id"] = img_id\n                self.items.append(entry_copy)\n                \n        # In-memory shard cache to minimize disk reads\n        self._shard_cache: Dict[str, Dict[str, torch.Tensor]] = {}\n\n    def __len__(self) -> int:\n        return len(self.items)\n\n    def _get_shard_tensors(self, shard_name: str) -> Dict[str, torch.Tensor]:\n        """Lazy load and cache safetensors shard file."""\n        if shard_name not in self._shard_cache:\n            shard_path = self.cache_dir / shard_name\n            if not shard_path.exists():\n                raise FileNotFoundError(f"Shard file {shard_path} does not exist.")\n            self._shard_cache[shard_name] = load_file(shard_path)\n        return self._shard_cache[shard_name]\n\n    def _load_resized_mask(\n        self,\n        rel_mask_path: Optional[str],\n        grid_shape: Tuple[int, int]\n    ) -> Optional[torch.Tensor]:\n        """Load ground truth binary mask and resize to patch grid resolution (0/1)."""\n        if not rel_mask_path:\n            return None\n            \n        paths = get_paths()\n        full_path = paths.project_root / rel_mask_path\n        if not full_path.exists():\n            full_path = paths.raw_data / rel_mask_path\n            if not full_path.exists():\n                return None\n                \n        try:\n            with Image.open(full_path) as m_img:\n                m_arr = (np.array(m_img.convert("L")) > 0).astype(np.float32)\n                \n            gh, gw = grid_shape\n            # Resize using PIL nearest-neighbor to preserve binary labels\n            m_pil = Image.fromarray((m_arr * 255).astype(np.uint8))\n            m_resized = m_pil.resize((gw, gh), Image.Resampling.NEAREST)\n            m_bin = (np.array(m_resized) > 0).astype(np.float32)\n            return torch.from_numpy(m_bin)\n        except Exception as e:\n            logger.warning(f"Failed to load mask {rel_mask_path}: {e}")\n            return None\n\n    def __getitem__(self, idx: int) -> Dict[str, Any]:\n        """Fetch item by index.\n\n        Returns:\n            Dict containing image_id, cls_token, mean_patch_token, patch_tokens,\n            grid_shape, optional lesion_mask, optional leaf_mask, and metadata.\n        """\n        item = self.items[idx]\n        img_id = item["image_id"]\n        shard_name = item["shard"]\n        grid_shape = tuple(item["grid_shape"])  # (gh, gw)\n        \n        tensors = self._get_shard_tensors(shard_name)\n        \n        cls_token = tensors.get(f"{img_id}_cls")\n        mean_patch_token = tensors.get(f"{img_id}_mean")\n        patch_tokens = tensors.get(f"{img_id}_patch")\n        \n        if cls_token is None or patch_tokens is None:\n            raise KeyError(f"Feature tensors for image \'{img_id}\' not found in shard \'{shard_name}\'.")\n\n        # Load ground truth masks if available\n        lesion_mask = self._load_resized_mask(item.get("rel_lesion_mask_path"), grid_shape)\n        leaf_mask = self._load_resized_mask(item.get("rel_leaf_mask_path"), grid_shape)\n\n        sample = {\n            "image_id": img_id,\n            "cls_token": cls_token,\n            "mean_patch_token": mean_patch_token if mean_patch_token is not None else patch_tokens.mean(dim=(0, 1)),\n            "patch_tokens": patch_tokens,\n            "grid_shape": torch.tensor(grid_shape, dtype=torch.int64),\n            "lesion_mask": lesion_mask if lesion_mask is not None else torch.zeros(grid_shape, dtype=torch.float32),\n            "leaf_mask": leaf_mask if leaf_mask is not None else torch.ones(grid_shape, dtype=torch.float32),\n            "metadata": {\n                "source": item.get("source", "unknown"),\n                "split": item.get("split", "train"),\n                "severity": item.get("severity", 0.0),\n                "cluster_id": item.get("cluster_id", 0),\n                "original_size": item.get("original_size", [448, 448]),\n                "padded_size": item.get("padded_size", [448, 448])\n            }\n        }\n\n        if self.transform is not None:\n            sample = self.transform(sample)\n\n        return sample\n')
    print("Inline pipeline modules written successfully to /content/ml/")


In [ ]:
# Step 3: Run Storage Footprint Estimator
import sys, importlib
if 'ml.features.storage_estimate' in sys.modules:
    import ml.features.storage_estimate
    importlib.reload(ml.features.storage_estimate)
from ml.features.storage_estimate import print_storage_summary
print_storage_summary(static_n=1000, eth_n=12520, available_gb=15.0)


In [ ]:
# Step 4: Extract & Cache Features for Static In-The-Wild Datasets
import sys, importlib, pandas as pd, torch
from pathlib import Path
if 'ml.features.extract_features' in sys.modules:
    import ml.features.extract_features
    importlib.reload(ml.features.extract_features)
from ml.features.extract_features import run_feature_extraction_pipeline
from ml.utils.paths import get_paths
from ml.utils.io import read_parquet

paths = get_paths()
candidates = [
    paths.processed_data / 'static' / 'manifest.parquet',
    paths.data_root / 'processed' / 'static' / 'manifest.parquet',
    paths.project_root / 'data' / 'processed' / 'static' / 'manifest.parquet',
    Path('/content/data/processed/static/manifest.parquet')
]

static_df = pd.DataFrame()
for p in candidates:
    if p.exists():
        df = read_parquet(p)
        if 'rel_image_path' in df.columns or 'path' in df.columns or 'rel_mask_path' in df.columns:
            static_df = df
            print(f'Loaded static manifest with {len(static_df)} images from {p}.')
            break

if static_df.empty:
    print('Static manifest missing or outdated. Generating static manifest automatically...')
    try:
        if 'ml.data.build_static_manifest' in sys.modules:
            import ml.data.build_static_manifest
            importlib.reload(ml.data.build_static_manifest)
        from ml.data.build_static_manifest import process_static_manifest
        static_df = process_static_manifest()
        print(f'Built static manifest with {len(static_df)} images.')
    except Exception as e:
        print(f'Could not build static manifest automatically: {e}. Running feature extraction on empty manifest.')

use_gpu = torch.cuda.is_available()
print('CUDA GPU Available:', use_gpu)
static_res = run_feature_extraction_pipeline(
    manifest_df=static_df,
    backbone_id='facebook/dinov2-small',
    resolution=448,
    use_gpu=use_gpu,
    is_stub=not use_gpu
)
print('Static Feature Extraction Complete:', static_res)


In [ ]:
# Step 5: Extract & Cache Features for ETH Zurich Wheat Time Series
import sys, importlib, pandas as pd, torch
from pathlib import Path
if 'ml.features.extract_features' in sys.modules:
    import ml.features.extract_features
    importlib.reload(ml.features.extract_features)
from ml.features.extract_features import run_feature_extraction_pipeline
from ml.utils.paths import get_paths
from ml.utils.io import read_parquet

paths = get_paths()
eth_candidates = [
    paths.processed_data / 'eth' / 'images_long.parquet',
    paths.data_root / 'processed' / 'eth' / 'images_long.parquet',
    paths.project_root / 'data' / 'processed' / 'eth' / 'images_long.parquet',
    Path('/content/data/processed/eth/images_long.parquet')
]

eth_df = pd.DataFrame()
for p in eth_candidates:
    if p.exists():
        df = read_parquet(p)
        if 'rel_image_path' in df.columns or 'path' in df.columns or 'rel_mask_path' in df.columns:
            eth_df = df
            print(f'Loaded ETH sequence manifest with {len(eth_df)} image frames from {p}.')
            break

if eth_df.empty:
    print('ETH sequence manifest missing or outdated. Generating ETH manifest automatically...')
    try:
        if 'ml.data.eth_wheat' in sys.modules:
            import ml.data.eth_wheat
            importlib.reload(ml.data.eth_wheat)
        from ml.data.eth_wheat import process_eth_dataset
        process_eth_dataset()
        for p in eth_candidates:
            if p.exists():
                df = read_parquet(p)
                if 'rel_image_path' in df.columns or 'path' in df.columns or 'rel_mask_path' in df.columns:
                    eth_df = df
                    break
        print(f'Built ETH sequence manifest with {len(eth_df)} image frames.')
    except Exception as e:
        print(f'Could not build ETH sequence manifest automatically: {e}. Running feature extraction on empty manifest.')

use_gpu = torch.cuda.is_available()
eth_res = run_feature_extraction_pipeline(
    manifest_df=eth_df,
    backbone_id='facebook/dinov2-small',
    resolution=448,
    eth_mode='downscaled_crop',
    use_gpu=use_gpu,
    is_stub=not use_gpu
)
print('ETH Feature Extraction Complete:', eth_res)


In [ ]:
# Step 6: Compute OOD Statistics & Mahalanobis Distance Percentiles
if 'ml.features.ood_stats' in sys.modules:
    import ml.features.ood_stats
    importlib.reload(ml.features.ood_stats)
from ml.features.ood_stats import process_ood_stats

cache_dir = Path(static_res['cache_dir'])
ood_res = process_ood_stats(cache_dir, gamma=0.01, percentiles=[95.0, 99.0])
print('OOD Statistics Calculation Complete:')
print('Embedding Dimension:', ood_res.get('embed_dim'))
print('Validation Mahalanobis Percentiles:', ood_res.get('percentiles'))

In [ ]:
# Step 7: Verify PyTorch CachedFeatureDataset Reading
from ml.features.feature_dataset import CachedFeatureDataset

ds = CachedFeatureDataset(cache_dir=cache_dir)
print(f'Cached Feature Dataset initialized with {len(ds)} items.')
if len(ds) > 0:
    sample = ds[0]
    print('Sample Image ID:', sample['image_id'])
    print('CLS Token Shape:', sample['cls_token'].shape)
    print('Patch Tokens Shape:', sample['patch_tokens'].shape)
    print('Grid Shape:', sample['grid_shape'].tolist())

In [ ]:
# Step 8: Compress Feature Cache & Download Summary
import shutil
from google.colab import files

if cache_dir.exists():
    print(f'Compressing feature cache directory {cache_dir}...')
    zip_path = shutil.make_archive('/content/phase04_features_cache', 'zip', str(cache_dir))
    print('Feature Cache Zip created at:', zip_path)
    print('Triggering browser download...')
    files.download(zip_path)
else:
    print('Cache directory not found.')